# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)

### machine learning (scikit-learn)
import pandas as pd
from torch.utils.data import DataLoader

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:

import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps
import smartcheck.deep_learning_project_specific as mps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

#### Feature selection (retour d'expérience sur VIF d'autre notebook pour même cible de comparaison)

In [ ]:
col_to_drop1 = [
    'date_et_heure_de_comptage_day_of_year',
    'date_et_heure_de_comptage_year',
    'date_et_heure_de_comptage_month',
    'date_et_heure_de_comptage_day',
    'date_et_heure_de_comptage_day_of_week',
    'date_et_heure_de_comptage_hour'
]

In [ ]:
col_to_drop2 = [
    'latitude',
    'longitude',
    'date_et_heure_de_comptage_sin_month',
    'date_et_heure_de_comptage_sin_week',
]

In [ ]:
col_to_drop3 = [
    'date_et_heure_de_comptage_cos_month',
]

In [ ]:
df = cast(pd.DataFrame, df.drop(columns=col_to_drop1+col_to_drop2+col_to_drop3))
df_feat_sel = df.copy()

# 3. Regression modeling

In [ ]:
dataset = HourlyCounterDataset(
    y_series, exog_array, input_window=24, forecast_horizon=6
)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for x_y, x_exo, y_target in loader:
    preds = model(x_y, x_exo)
    loss = loss_fn(preds, y_target)